# IMU Sign Confusion Circos Diagram

Render circos-style sign-confusion diagrams from the per-fold confusion matrix pickles (`cm_*.pkl`) produced by `imu/scripts/grid_search.py`. Uses `correspondance.json` (generated by `sincronizacion.ipynb`) to map numeric label indices to human-readable Spanish sign names.

## Configuration

In [ ]:
CORRESPONDANCE_PATH = "correspondance.json"
CMAP = "Paired"


## Build circos diagrams

In [ ]:
import pickle
import pandas as pd
import json
from pycirclize import Circos
import glob
import seaborn as sns
import numpy as np


names = json.load(open(CORRESPONDANCE_PATH,"r"))
names_loso = {k:v for k,v in names.items() if k != "42"}
print(names_loso)
cmap=CMAP

print(names)
def clean(val):
    return f"¿{val.split('_')[0].upper()}?" if "_" in val else val.upper()

row_names = list(map(clean,names.values()))
row_names_loso = list(map(clean,names_loso.values()))

# General partition

for db_type in ["temporal"]:

    """with open(f"cm_rf_{db_type}_balanced.pkl", "rb") as f:
        cm = pickle.load(f)
        matrix_df = pd.DataFrame(cm, index=row_names, columns=row_names)
        circos = Circos.chord_diagram(
            matrix_df,
            space=3,
            cmap=cmap,
            label_kws=dict(size=12,orientation="vertical"),
            link_kws=dict(ec="black", lw=0.5, direction=1),
        )

        circos.savefig(f"imu_{db_type}_balanced.png")
"""
    agg = None
    for db in glob.glob(f"cm_rf_{db_type}_loso_*.pkl"):

        with open(db, "rb") as f:
            cm = pickle.load(f)
        
        if agg is None:
            agg = cm
        else: 
            try:
                agg = agg + cm
            except ValueError:
                cm_cleaned = np.delete(cm, 42, axis=0)  # Remove row 42
                cm_cleaned = np.delete(cm_cleaned, 42, axis=1)  # Remove column 42
                agg = agg + cm_cleaned
                print(f"Error adding confusion matrices for {db}")
                continue

    print(agg)

    np.fill_diagonal(agg, 0)

    matrix_df = pd.DataFrame(agg, index=row_names_loso, columns=row_names_loso)
    print(matrix_df)
    circos = Circos.chord_diagram(
        matrix_df,
        space=3,
        label_kws=dict(size=12,orientation="vertical"),
        link_kws=dict(ec="black", lw=0.5, direction=1),
        order="desc"
    )

    circos.savefig(f"imu_{db_type}_loso_confmap.png",dpi=600)

    #DEAF

    with open(f"cm_rf_{db_type}_deaf.pkl", "rb") as f:
        cm = pickle.load(f)
        matrix_df = pd.DataFrame(cm, index=["Signant","Non-signant"], columns=["Signant","Non-signant"])

        import matplotlib.pyplot as plt

        plt.figure(figsize=(6, 5))
        sns.heatmap(matrix_df, annot=True, fmt="d", cmap="Blues", cbar=False)
        plt.title("Confusion Matrix")
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.savefig(f"imu_{db_type}_deaf_cm.png")

    

